# Synthèse multi-scénarios

Décomposition sectorielle des empreintes environnementales (limites planétaires,
LP) par sous-processus de consommation, pour le scénario de base et les
scénarios de transition 2050.

Génère 3 types de figures dans `figures/` :
- 1 barre empilée en % par scénario (`scenario_NN_<nom>.png`)
- 1 figure de synthèse tous scénarios, normalisée en % (`synthesis_all_scenarios.png`)
- 1 figure de synthèse tous scénarios, valeurs absolues avec seuils LP/DLS (`synthesis_by_lp_all_scenarios.png`)

Logique de calcul et de tracé partagée avec les autres notebooks via le
package [`planefr_lib`](./planefr_lib/).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
if r"C:\Users\Arnaud\Documents\PlaneFR\notebooks" not in sys.path:
    sys.path.insert(0, r"C:\Users\Arnaud\Documents\PlaneFR\notebooks")

import matplotlib.pyplot as plt

from planefr_lib import config, io, processing
from planefr_lib.plot_stacked_bar import create_stacked_bar_chart, create_synthesis_figure, create_synthesis_figure_by_lp

Failed to read module file 'c:\Users\Arnaud\AppData\Local\Programs\Python\Python314\Lib\functools.py' for module 'functools': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\Arnaud\AppData\Roaming\Python\Python314\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "C:\Users\Arnaud\AppData\Roaming\Python\Python314\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "c:\Users\Arnaud\AppData\Local\Programs\Python\Python314\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1335, in _find_

## 1. Chargement des données communes

In [2]:
facteurs_carac_df = io.load_facteurs_carac()
bridge_matrices_df = io.load_bridge_matrices()
seuils_df = io.load_seuils()
dls_df = io.load_dls()

scenario_folders = config.get_scenario_folders(exclude_2019=True)
scenario_names = [f.name for f in scenario_folders]
print(f"Scénarios trouvés ({len(scenario_folders)}): {scenario_names}")

Scénarios trouvés (4): ['Base_year_2015', 'Sufficiency_NZE_2050', 'Tech_NZE_2050', 'TREND_2050']


## 2. Traitement de tous les scénarios

In [3]:
all_scenarios_data = []
all_subprocess_to_lp = []

for scenario_folder in scenario_folders:
    print(f"Traitement : {scenario_folder.name}")
    data_by_subprocess, subprocess_to_lp = processing.process_scenario(
        scenario_folder, facteurs_carac_df, bridge_matrices_df, seuils_df
    )
    all_scenarios_data.append(data_by_subprocess)
    all_subprocess_to_lp.append(subprocess_to_lp)
    print(f"  {len(data_by_subprocess)} sous-processus traités")

assert len(all_scenarios_data) == len(scenario_folders), "Certains scénarios n'ont produit aucune donnée"
print(f"\n{len(all_scenarios_data)}/{len(scenario_folders)} scénarios traités avec succès.")

Traitement : Base_year_2015
  7 sous-processus traités
Traitement : Sufficiency_NZE_2050
  7 sous-processus traités
Traitement : Tech_NZE_2050
  7 sous-processus traités
Traitement : TREND_2050
  7 sous-processus traités

4/4 scénarios traités avec succès.


## 3. Figure par scénario (barres empilées, %)

In [4]:
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

for scenario_idx, (scenario_folder, data_by_subprocess, subprocess_to_lp) in enumerate(
    zip(scenario_folders, all_scenarios_data, all_subprocess_to_lp)
):
    fig, ax = create_stacked_bar_chart(
        data_by_subprocess, seuils_df, subprocess_to_lp,
        scenario_name=scenario_folder.name,
    )
    output_path = config.FIGURES_DIR / f"scenario_{scenario_idx:02d}_{scenario_folder.name}.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Sauvegardé : {output_path.name}")

Sauvegardé : scenario_00_Base_year_2015.png
Sauvegardé : scenario_01_Sufficiency_NZE_2050.png
Sauvegardé : scenario_02_Tech_NZE_2050.png
Sauvegardé : scenario_03_TREND_2050.png


## 4. Figure de synthèse (normalisée en %)

In [5]:
fig_synthesis, _ = create_synthesis_figure(all_scenarios_data, seuils_df, all_subprocess_to_lp, scenario_names)

output_path = config.FIGURES_DIR / "synthesis_all_scenarios.png"
fig_synthesis.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close(fig_synthesis)
print(f"Sauvegardé : {output_path.name}")

C:\Users\Arnaud\Documents\PlaneFR\notebooks\planefr_lib\plot_stacked_bar.py:196: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.03, 1, 0.96])


Sauvegardé : synthesis_all_scenarios.png


## 5. Figure de synthèse alternative (par LP, valeurs absolues + seuils)

In [6]:
fig_by_lp, _ = create_synthesis_figure_by_lp(all_scenarios_data, seuils_df, all_subprocess_to_lp, scenario_names, dls_df)

output_path = config.FIGURES_DIR / "synthesis_by_lp_all_scenarios.png"
fig_by_lp.savefig(output_path, dpi=300, bbox_inches="tight")
plt.close(fig_by_lp)
print(f"Sauvegardé : {output_path.name}")

Sauvegardé : synthesis_by_lp_all_scenarios.png
